In [1]:
import pandas as pd

In [2]:
CORPUS_URL='https://github.com/Magallanes-at-UTDT/texts/raw/main/data/processed/RawCorpus.csv'
corpus = pd.read_csv(CORPUS_URL)

In [8]:
corpus["text_raw"].str.len().describe()

count         6.000000
mean     205020.333333
std      137228.846929
min       40765.000000
25%       92724.500000
50%      218343.000000
75%      301006.750000
max      373258.000000
Name: text_raw, dtype: float64

In [3]:
corpus.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   party         6 non-null      object
 1   text_raw      6 non-null      object
 2   n_characters  6 non-null      int64 
 3   n_words       6 non-null      int64 
dtypes: int64(2), object(2)
memory usage: 324.0+ bytes


Each row represents one political party's manifesto.

The corpus contains four variables:

| Variable       | Description                           |
| -------------- | ------------------------------------- |
| `party`        | Political party.                      |
| `text_raw`     | Original text extracted from the PDF. |
| `n_characters` | Number of characters in the document. |
| `n_words`      | Number of words in the original text. |

The variable `text_raw` is the starting point for the remainder of this notebook. Although the text is readable by humans, it is not yet suitable for quantitative analysis. Our goal is to transform these documents into a **Document-Term Matrix (DTM)**



## The Document-Term Matrix

Most quantitative text analysis methods do not operate directly on text. Instead, they require a **Document-Term Matrix (DTM)**.

In a DTM:

* each **row** represents a document;
* each **column** represents a unique lexical feature (word or normalized token);
* each **cell** contains the frequency with which that feature appears in the document.

For example:

| Document | economy | education | health | tax |
| -------- | ------: | --------: | -----: | --: |
| Party A  |      12 |         8 |      5 |   3 |
| Party B  |       4 |        15 |      9 |   1 |
| Party C  |      18 |         2 |      7 |  10 |

This numerical representation allows statistical and machine learning algorithms to analyze textual data.

However, constructing a useful DTM requires deciding **which lexical forms become columns**. Different representations of the same concept (e.g., plurals, verb conjugations, OCR errors, or missing accents) unnecessarily increase the number of columns and make the matrix sparser.

In [4]:
from google import genai

print("Installation successful!")

Installation successful!


In [5]:
# Replace with your own API key

from pathlib import Path

API_KEY = Path("sometext.txt").read_text().strip()
client = genai.Client(api_key=API_KEY)


for model in client.models.list():
    print(model.name)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [6]:
MODEL="gemini-3.6-flash"
response = client.models.generate_content(
    model=MODEL,
    contents="Say hello in Spanish."
)

print(response.text)

¡Hola!


In [9]:
PROMPT = """
You are assisting in quantitative text analysis.

Your task is to convert the following document into a normalized lexical
representation suitable for constructing a Document-Term Matrix (DTM).

Return ONLY valid JSON with one field named "tokens".

Instructions

1. Convert all words to lowercase.
2. Remove punctuation.
3. Remove stopwords.
4. Convert verbs to their infinitive form.
5. Convert plural nouns to singular.
6. Preserve proper names and acronyms.
7. Preserve four-digit years (e.g., 1824, 1993, 2021). Discard all other numeric tokens.
8. Preserve multi-word named entities using underscores (e.g., united_states, european_union).
9. Keep only lexical content words (nouns, main verbs, adjectives, and adverbs).
10. Return only the JSON object. Do not include explanations, comments, or markdown.

Document

{text}
"""

In [10]:
import json
from google.genai import types


TOKEN_SCHEMA = {
    "type": "object",
    "properties": {
        "tokens": {
            "type": "array",
            "items": {"type": "string"}
        }
    },
    "required": ["tokens"]
}


def lexical_normalize(text):

    response = client.models.generate_content(
        model=MODEL,
        contents=PROMPT.format(text=text),
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
            response_schema=TOKEN_SCHEMA
        )
    )

    result = json.loads(response.text)

    return result["tokens"]

In [12]:
from tqdm.auto import tqdm

tqdm.pandas()

corpus["tokens"] = corpus["text_raw"].progress_apply(
    lexical_normalize
)

  0%|          | 0/6 [00:00<?, ?it/s]

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 3.284676375s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'}, 

In [ ]:
corpus[["party", "tokens"]]

In [13]:
corpus.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   party         6 non-null      object
 1   text_raw      6 non-null      object
 2   n_characters  6 non-null      int64 
 3   n_words       6 non-null      int64 
dtypes: int64(2), object(2)
memory usage: 324.0+ bytes


In [ ]:
# !pip install pyarrow #and restart kernel

In [ ]:
# from pathlib import Path

# PROCESSED_DIR = Path("data/processed")
# corpus.to_parquet(
#     PROCESSED_DIR / "CorpusWithLLMTokens.parquet"
# )


In [14]:


CORPUS_URL = (
    "https://github.com/Magallanes-at-UTDT/texts/"
    "raw/main/data/processed/CorpusWithLLMTokens.parquet"
)

corpus_llm = pd.read_parquet(CORPUS_URL)

In [15]:
corpus_llm["normalized_text"] = corpus_llm["tokens"].str.join(" ")

In [16]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

dtm_sparse = vectorizer.fit_transform(corpus_llm["normalized_text"])

In [17]:
import pandas as pd

dtm = pd.DataFrame(
    dtm_sparse.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=corpus["party"]
)

In [18]:
dtm.columns[:50]

Index(['00', '000', '001', '0270', '029', '03', '05', '052', '07148895', '079',
       '08', '09', '10', '100', '1000', '101', '11', '110', '118', '12', '120',
       '13', '130', '14', '1400', '147', '15', '158', '16', '163', '17', '18',
       '1824', '184', '19', '19315', '1950', '196', '1960', '1970', '1989',
       '1991', '1993', '1994', '1996', '20', '200', '2000', '2001', '2002'],
      dtype='object')

In [20]:
validYears = dtm.columns.str.fullmatch(r"(18|19|20)\d{2}")
textColumns = ~dtm.columns.str.isdigit()

dtm.loc[:, validYears | textColumns].columns

Index(['1824', '1950', '1960', '1970', '1989', '1991', '1993', '1994', '1996',
       '2000',
       ...
       'ético', 'étnico', 'éxito', 'índice', 'óptimo', 'órgano',
       'órgano_de_control', 'últimamente', 'último', 'únicamente'],
      dtype='object', length=3808)

In [21]:
dtm=dtm.loc[:, validYears | textColumns]

In [25]:
[word for word in dtm.columns if word.startswith("econ")]

['economía', 'economía_de_escala', 'economía_mixta', 'económico']

In [27]:
[word for word in dtm.columns if word.startswith("polít")]

['política',
 'política_nacional_de_cultura_al_2030',
 'política_nacional_de_gestión_logística_y_abastecimiento_en_salud',
 'política_nacional_de_infraestructura_y_equipamiento',
 'política_nacional_de_integración_funcional_del_sistema_de_salud',
 'política_nacional_de_juventud',
 'político']

In [ ]:
from pathlib import Path

PROCESSED_DIR = Path("data/processed")


dtm.to_parquet(
    PROCESSED_DIR / "dtm_counts.parquet",
    index=True
)